# Activation-continuation: Colab execution gate

This notebook is intentionally thin. It mounts Drive and calls versioned, importable repository modules; it contains no benchmark-selection, generation, probing, or analysis logic.

Run cells in order. A failed preflight is a hard stop: do not edit the protocol, shorten the qualification, use a different model/precision, or run benchmark generation.

In [ ]:
from pathlib import Path

# Paste the exact reviewed commit reported by Codex after pushing the package.
REPOSITORY_URL = 'https://github.com/mangesh-ux/reasoning-confidence-audit.git'
REPOSITORY_REF = 'REPLACE_WITH_REVIEWED_COMMIT_SHA'
PRIVATE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/reasoning_activation_study')
RUN_BENCHMARK_STUDY = False  # Must remain false for this portability handoff.

assert RUN_BENCHMARK_STUDY is False, 'Benchmark generation is not authorized in this notebook run.'
assert len(REPOSITORY_REF) == 40 and all(character in '0123456789abcdef' for character in REPOSITORY_REF), (
    'Set REPOSITORY_REF to the 40-character reviewed commit hash before continuing.'
)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
PRIVATE_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive mounted; private artifact root is ready.')

In [ ]:
import subprocess

REPOSITORY_DIR = Path('/content/reasoning-confidence-audit')

def normalize_repository_url(value):
    return value.strip().rstrip('/').removesuffix('.git')

if REPOSITORY_DIR.exists():
    if not (REPOSITORY_DIR / '.git').exists():
        raise RuntimeError(f'Refusing to reuse a non-Git directory: {REPOSITORY_DIR}')
    ORIGIN_URL = subprocess.check_output(
        ['git', '-C', str(REPOSITORY_DIR), 'remote', 'get-url', 'origin'], text=True
    ).strip()
    if normalize_repository_url(ORIGIN_URL) != normalize_repository_url(REPOSITORY_URL):
        raise RuntimeError('Existing checkout origin does not match REPOSITORY_URL.')
    if subprocess.check_output(['git', '-C', str(REPOSITORY_DIR), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('Existing checkout is dirty; use a clean checkout for provenance.')
    subprocess.run(['git', '-C', str(REPOSITORY_DIR), 'fetch', '--tags', '--prune', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
subprocess.run(['git', '-C', str(REPOSITORY_DIR), 'checkout', '--detach', REPOSITORY_REF], check=True)
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPOSITORY_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert SOURCE_COMMIT == REPOSITORY_REF, 'Checkout did not resolve to the reviewed commit hash.'
print(f'Repository checked out at {SOURCE_COMMIT}')

In [ ]:
import sys

ENVIRONMENT_DIR = Path('/content/activation-continuation-venv')
REQUIREMENTS = REPOSITORY_DIR / 'activation_continuation' / 'configs' / 'requirements-colab.txt'
if not ENVIRONMENT_DIR.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(ENVIRONMENT_DIR)], check=True)
VENV_PYTHON = ENVIRONMENT_DIR / 'bin' / 'python'
subprocess.run([str(VENV_PYTHON), '-m', 'pip', 'install', '--upgrade', 'pip==24.0'], check=True)
subprocess.run([str(VENV_PYTHON), '-m', 'pip', 'install', '--no-cache-dir', '-r', str(REQUIREMENTS)], check=True)
subprocess.run([str(VENV_PYTHON), '-m', 'pip', 'check'], check=True)
print('Isolated, fully pinned environment installed and dependency consistency check passed.')

In [ ]:
import os
import re

COLAB_ENVIRONMENT = os.environ.copy()
COLAB_ENVIRONMENT['PYTHONPATH'] = os.pathsep.join([
    str(REPOSITORY_DIR),
    str(REPOSITORY_DIR / 'activation_continuation' / 'src'),
])
PREFLIGHT = subprocess.run([
    str(VENV_PYTHON), '-m', 'activation_continuation_colab.preflight',
    '--config', str(REPOSITORY_DIR / 'activation_continuation' / 'configs' / 'colab_runtime.json'),
    '--private-artifact-root', str(PRIVATE_ARTIFACT_ROOT),
    '--execute-synthetic',
], env=COLAB_ENVIRONMENT, check=False, capture_output=True, text=True)
PREFLIGHT_EXIT_CODE = PREFLIGHT.returncode
print(PREFLIGHT.stdout)
REPORT_FILENAME_MATCH = re.search(
    r'^public_safe_report_filename: ([A-Za-z0-9_.-]+\.json)$', PREFLIGHT.stdout, flags=re.MULTILINE
)
assert REPORT_FILENAME_MATCH is not None, 'Preflight did not emit a public-safe report filename.'
PREFLIGHT_REPORT_FILENAME = REPORT_FILENAME_MATCH.group(1)
assert PREFLIGHT_EXIT_CODE == 0, (
    'STOP: Colab preflight did not pass. Do not run benchmark generation; return the PASS/FAIL table and public-safe report to Codex.'
)
print('Qualification passed. Benchmark execution remains disabled in this handoff.')

In [ ]:
import json

PUBLIC_SAFE_DIR = PRIVATE_ARTIFACT_ROOT / 'public_safe'
REPORT_FILE = PUBLIC_SAFE_DIR / PREFLIGHT_REPORT_FILENAME
assert REPORT_FILE.is_file(), f'Expected public-safe preflight report is missing: {PREFLIGHT_REPORT_FILENAME}'
PUBLIC_PREFLIGHT_REPORT = json.loads(REPORT_FILE.read_text(encoding='utf-8'))
print(json.dumps(PUBLIC_PREFLIGHT_REPORT, indent=2, sort_keys=True))
print('Return the JSON above, plus the PASS/FAIL table, to Codex before any future benchmark step.')

In [ ]:
# Deliberately disabled. The portable package provides a hard gate but no benchmark runner.
if RUN_BENCHMARK_STUDY:
    from activation_continuation_colab.benchmark_gate import (
        explain_no_benchmark_execution,
        verify_benchmark_gate,
    )
    from activation_continuation_colab.config import load_colab_runtime_config

    COLAB_CONFIG = load_colab_runtime_config(
        REPOSITORY_DIR / 'activation_continuation' / 'configs' / 'colab_runtime.json'
    )
    verify_benchmark_gate(
        config=COLAB_CONFIG,
        public_preflight_report=PUBLIC_PREFLIGHT_REPORT,
        repository_root=REPOSITORY_DIR,
    )
    raise RuntimeError(explain_no_benchmark_execution())
print('STOP: benchmark generation has not been invoked.')